#Estratégia 2 — Naive Bayes e Regressão Logística

##1. Objetivo
Nesta etapa, foram implementados dois modelos clássicos de aprendizado de máquina supervisionado: Naive Bayes (variante Multinomial) e Regressão Logística. Diferente da abordagem heurística, que se baseia em regras manuais estáticas, estes algoritmos buscam aprender os padrões probabilísticos e os pesos associados a cada termo do vocabulário gerado. O objetivo é testar o desempenho desses modelos utilizando as representações Bag-of-Words (BoW) e TF-IDF geradas no pré-processamento.

#2.[Carregamento do Dataset e Geração das Representações]
Como estamos partindo do arquivo CSV pré-processado, carregamos os dados e recriamos as representações vetoriais (Bag-of-Words e TF-IDF) utilizando os textos lematizados, limitando o vocabulário a 3.000 características.

In [ ]:
import pandas as pd
import pickle
from google.colab import files

print("Selecione os arquivos: X_bow.pkl, X_tfidf.pkl e labels.pkl")
uploaded = files.upload()

# Carrega as matrizes e rótulos gerados pela Pessoa 1
with open('X_bow.pkl', 'rb') as f:
    X_bow = pickle.load(f)

with open('X_tfidf.pkl', 'rb') as f:
    X_tfidf = pickle.load(f)

with open('labels.pkl', 'rb') as f:
    y = pickle.load(f)

print("Dados carregados com sucesso!")
print(f"Dimensão Bag-of-Words: {X_bow.shape}")
print(f"Dimensão TF-IDF: {X_tfidf.shape}")
print(f"Distribuição de classes:\n{y.value_counts()}")

Selecione os arquivos: X_bow.pkl, X_tfidf.pkl e labels.pkl


Saving labels.pkl to labels.pkl
Saving X_bow.pkl to X_bow.pkl
Saving X_tfidf.pkl to X_tfidf.pkl
Dados carregados com sucesso!
Dimensão Bag-of-Words: (450, 3000)
Dimensão TF-IDF: (450, 3000)
Distribuição de classes:
classe_binaria
Spam    375
Ham      75
Name: count, dtype: int64


#3. Configuração da Validação Cruzada Estratificada
Para garantir a robustez da avaliação em um conjunto de dados pequeno (450 instâncias), adotou-se a validação cruzada estratificada (Stratified K-Fold) com 5 partições ($k=5$). Isso assegura que a proporção original de desbalanceamento (aproximadamente 83% Spam e 17% Ham) seja rigorosamente mantida em todas as dobras de treino e teste.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Configuração do K-Fold estratificado (k=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#4. Definição dos Modelos e Estratégias de Balanceamento
Foram definidos o Multinomial Naive Bayes e a Regressão Logística, cada um em versão padrão e versão balanceada. Para a Regressão Logística, o balanceamento é feito via class_weight='balanced', que penaliza mais os erros cometidos na classe minoritária (Ham). Para o Naive Bayes, como esse hiperparâmetro não existe, o balanceamento foi implementado ajustando manualmente o class_prior para uma distribuição uniforme ([0.5, 0.5]), contrapondo a probabilidade a priori naturalmente enviesada para Spam que o modelo aprenderia da distribuição real das classes.
Além disso,  foi realizada uma busca via GridSearchCV para cada modelo: o parâmetro alpha (suavização de Laplace) foi testado para o Naive Bayes nos valores [0.1, 0.5, 1.0, 2.0], e o parâmetro C (inverso da regularização) foi testado para a Regressão Logística nos valores [0.01, 0.1, 1.0, 10.0]. A métrica de otimização escolhida foi o F1-Score macro, por ser mais sensível ao desempenho na classe minoritária do que a acurácia

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Grade de hiperparâmetros a testar
param_grid_nb = {'alpha': [0.1, 0.5, 1.0, 2.0]}
param_grid_lr = {'C': [0.01, 0.1, 1.0, 10.0]}

# class_prior "balanceado": prioriza a classe minoritária (Ham) na proporção inversa
prior_balanceado = [0.5, 0.5]  # prior uniforme, contrapondo o desbalanceamento natural do fit

# Modelos base
modelos_base = {
    'NaiveBayes_BoW_Padrao':        (MultinomialNB(), X_bow),
    'NaiveBayes_TFIDF_Padrao':      (MultinomialNB(), X_tfidf),
    'NaiveBayes_BoW_Balanceado':    (MultinomialNB(class_prior=prior_balanceado), X_bow),
    'NaiveBayes_TFIDF_Balanceado':  (MultinomialNB(class_prior=prior_balanceado), X_tfidf),
    'LogReg_BoW_Padrao':            (LogisticRegression(max_iter=1000, random_state=42), X_bow),
    'LogReg_TFIDF_Padrao':          (LogisticRegression(max_iter=1000, random_state=42), X_tfidf),
    'LogReg_BoW_Balanceado':        (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), X_bow),
    'LogReg_TFIDF_Balanceado':      (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), X_tfidf),
}

# Define qual grade de parâmetros usar para cada família de modelo
def get_param_grid(nome_modelo):
    if 'NaiveBayes' in nome_modelo:
        return param_grid_nb
    return param_grid_lr

#5. Treinamento, Predição e Avaliação
Nesta etapa, para cada um dos oito modelos definidos (NB e LogReg × BoW e TF-IDF × padrão e balanceado), primeiro executamos o GridSearchCV para selecionar o melhor hiperparâmetro dentro da própria validação cruzada estratificada, e em seguida geramos as predições finais via cross_val_predict usando o modelo já ajustado com o hiperparâmetro escolhido. As predições são avaliadas através das métricas de Precisão, Recall, F1-Score (por classe e macro) e da Matriz de Confusão, e cada resultado é armazenado em uma tabela consolidada para facilitar a comparação entre os modelos.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

df_predicoes_ml = pd.DataFrame({'Rotulo_Real': y.reset_index(drop=True)})
resultados_metricas = []  # lista para consolidar métricas por modelo

print("--- RESULTADOS DOS EXPERIMENTOS ---\n")

for nome, (modelo, X) in modelos_base.items():
    print(f"\n>> Avaliando modelo: {nome}")

    # Busca do melhor hiperparâmetro via GridSearchCV (otimizando F1-macro)
    grid = GridSearchCV(
        modelo,
        get_param_grid(nome),
        scoring='f1_macro',
        cv=skf
    )
    grid.fit(X, y)
    melhor_modelo = grid.best_estimator_
    print(f"Melhor hiperparâmetro encontrado: {grid.best_params_}")

    # Predições finais via validação cruzada com o modelo já ajustado
    y_pred = cross_val_predict(melhor_modelo, X, y, cv=skf)
    df_predicoes_ml[f'Pred_{nome}'] = y_pred

    relatorio = classification_report(y, y_pred, target_names=['Ham', 'Spam'], output_dict=True)
    print(classification_report(y, y_pred, target_names=['Ham', 'Spam']))

    cm = confusion_matrix(y, y_pred, labels=['Ham', 'Spam'])
    print("Matriz de Confusão (Linha=Real, Coluna=Previsto):")
    print(pd.DataFrame(cm, index=['Real_Ham', 'Real_Spam'], columns=['Prev_Ham', 'Prev_Spam']))
    print("-" * 65)

    # Consolida as métricas desse modelo em uma linha
    resultados_metricas.append({
        'Modelo': nome,
        'Melhor_Hiperparametro': str(grid.best_params_),
        'Precisao_Ham': relatorio['Ham']['precision'],
        'Recall_Ham': relatorio['Ham']['recall'],
        'F1_Ham': relatorio['Ham']['f1-score'],
        'Precisao_Spam': relatorio['Spam']['precision'],
        'Recall_Spam': relatorio['Spam']['recall'],
        'F1_Spam': relatorio['Spam']['f1-score'],
        'F1_Macro': relatorio['macro avg']['f1-score'],
        'Acuracia': relatorio['accuracy'],
        'VN_Ham': cm[0][0], 'FP_Ham': cm[0][1],
        'FN_Spam': cm[1][0], 'VP_Spam': cm[1][1],
    })

df_metricas = pd.DataFrame(resultados_metricas).sort_values('F1_Macro', ascending=False)
print("\n--- RESUMO COMPARATIVO (ordenado por F1-Macro) ---")
print(df_metricas[['Modelo', 'Melhor_Hiperparametro', 'F1_Macro', 'Recall_Ham', 'Precisao_Ham']])

--- RESULTADOS DOS EXPERIMENTOS ---


>> Avaliando modelo: NaiveBayes_BoW_Padrao
Melhor hiperparâmetro encontrado: {'alpha': 0.1}
              precision    recall  f1-score   support

         Ham       0.26      0.53      0.35        75
        Spam       0.88      0.69      0.77       375

    accuracy                           0.66       450
   macro avg       0.57      0.61      0.56       450
weighted avg       0.78      0.66      0.70       450

Matriz de Confusão (Linha=Real, Coluna=Previsto):
           Prev_Ham  Prev_Spam
Real_Ham         40         35
Real_Spam       116        259
-----------------------------------------------------------------

>> Avaliando modelo: NaiveBayes_TFIDF_Padrao
Melhor hiperparâmetro encontrado: {'alpha': 0.1}
              precision    recall  f1-score   support

         Ham       0.31      0.21      0.25        75
        Spam       0.85      0.91      0.88       375

    accuracy                           0.79       450
   macro avg       0.

#6. Exportação das Predições

As predições geradas por todos os modelos testados são armazenadas e exportadas para um arquivo CSV. Esse arquivo será essencial para a etapa final do projeto, onde se fará a consolidação e a análise comparativa de todas as estratégias desenvolvidas pelo grupo.
Dois arquivos são gerados: **predicoes_naivebayes_logreg.csv**, com as classificações de cada modelo para cada mensagem, e m**etricas_naivebayes_logreg**.csv, com um resumo consolidado (precisão, recall e F1-Score por classe, F1-Score macro, acurácia, hiperparâmetro escolhido e contagens da matriz de confusão) para cada um dos oito modelos testados

In [ ]:
nome_arquivo_predicoes = 'predicoes_naivebayes_logreg.csv'
nome_arquivo_metricas = 'metricas_naivebayes_logreg.csv'

df_predicoes_ml.to_csv(nome_arquivo_predicoes, index=False)
df_metricas.to_csv(nome_arquivo_metricas, index=False)

print(f"Sucesso! Arquivos '{nome_arquivo_predicoes}' e '{nome_arquivo_metricas}' gerados.")

files.download(nome_arquivo_predicoes)
files.download(nome_arquivo_metricas)


Sucesso! Arquivos 'predicoes_naivebayes_logreg.csv' e 'metricas_naivebayes_logreg.csv' gerados.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#7. Análise dos Resultados e Dificuldades Encontradas

# 7. Análise dos Resultados

O ajuste de hiperparâmetros e o balanceamento do Naive Bayes mudaram bastante os resultados da etapa anterior.

## Naive Bayes: o alpha resolveu o colapso
Antes, com `alpha=1.0` fixo, o NB com TF-IDF classificava 100% como Spam (Recall Ham = 0,00). Com o grid search, `alpha=0.1` foi o valor ótimo na maioria dos casos, elevando o Recall de Ham para 0,21–0,53 mesmo sem balanceamento.

## Efeito do balanceamento (class_prior)
- **BoW**: pouco ganho — padrão (Recall 0,53) ficou próximo do balanceado (Recall 0,51).
- **TF-IDF**: ganho relevante — Recall de Ham subiu de 0,21 (padrão) para 0,48 (balanceado), com mais falsos positivos.

## Melhor modelo geral
`LogReg_TFIDF_Balanceado` (`C=0.1`) teve o maior **F1-macro (0,66)** e a melhor Precisão de Ham entre os balanceados (0,52) — superando o `LogReg_BoW_Balanceado` (0,65) da etapa anterior.

## Trade-off Precisão x Recall
- `LogReg_TFIDF_Padrao` (`C=10`): Precisão Ham alta (0,75), mas Recall baixo (0,08).
- `LogReg_BoW_Balanceado`: Recall alto (0,55), Precisão menor (0,38).

Nenhum modelo atinge Precisão e Recall altos simultaneamente para Ham — reflexo do dataset pequeno (75 exemplos) e do desbalanceamento severo.

## Conclusão
**Regressão Logística + TF-IDF Balanceado (C=0,1)** foi a melhor configuração (F1-macro = 0,66), confirmando o valor de buscar hiperparâmetros mesmo em datasets pequenos.